# 4.3 · 多项式回归 / Polynomial Regression

> **课程定位 / Where this fits**
> 第 3 课，**Part 4 · 监督学习：回归**。
> Lesson 3, **Part 4 · Supervised Regression**.
>
> 线性回归只能画直线/平面，碰到弯曲的关系就无能为力。**多项式回归**用一个巧妙的技巧——把 $x, x^2, x^3\dots$ 都当作特征——让**线性模型拟合曲线**。更重要的是，它是讲清**过拟合**和**偏差-方差权衡**的最佳载体，这两个概念是整个机器学习的核心。
> Linear regression only fits lines/planes and fails on curved relationships. **Polynomial regression** uses a clever trick — treating $x, x^2, x^3\dots$ as features — to let a **linear model fit curves**. More importantly, it's the best vehicle for explaining **overfitting** and the **bias-variance trade-off**, the heart of all ML.
>
> 💼 **实战/面试视角**："什么是过拟合 / 偏差方差权衡 / 怎么诊断" 是 ML 面试**最核心**的概念题。
> 💼 **Practical/interview angle:** "what is overfitting / bias-variance / how to diagnose" are *the* core ML concept questions.

> 💡 **面试相关 / Interview-relevant**
> - "过拟合/欠拟合的表现与区别"（出镜率 ★★★★★）
> - "偏差-方差权衡是什么 / 谁随复杂度怎么变"（★★★★★）
> - "怎么诊断过拟合（train vs CV 差距）"（★★★★★）
> - "验证曲线 vs 学习曲线分别看什么"（★★★★★）
> - "高偏差 vs 高方差，分别怎么救"（★★★★★）

---

## 学习目标 / Learning Objectives

1. 理解多项式回归 = 线性模型 + 多项式特征。
   Understand polynomial regression = linear model + polynomial features.
2. 直观看清**欠拟合 / 恰好 / 过拟合**。
   See underfitting / just-right / overfitting visually.
3. 用实验拆解**偏差² 与方差**随复杂度的变化。
   Decompose bias² and variance versus complexity empirically.
4. 用**验证曲线**选复杂度。
   Use the validation curve to choose complexity.
5. 用**学习曲线**区分高偏差/高方差，并知道各自怎么救。
   Use the learning curve to tell high-bias from high-variance and how to fix each.

## 目录 / TOC
1. [先建直觉 + 数据](#1)
2. [欠拟合/恰好/过拟合 ⭐](#2)
3. [偏差-方差权衡（实验拆解）⭐](#3)
4. [验证曲线：选复杂度 ⭐](#4)
5. [学习曲线：诊断 + 对症下药 ⭐](#5)
6. [小结](#6)


<a id="1"></a>
## 1. 先建直觉 + 数据 / Intuition & Data

线性模型 $\hat y = w_0 + w_1 x$ 只能画直线。但如果我们**先把 $x$ 扩展成 $[x, x^2, x^3]$，再做线性回归**，模型就变成 $\hat y = w_0 + w_1 x + w_2 x^2 + w_3 x^3$——一条曲线！**关键洞察：模型对参数 $\mathbf{w}$ 仍是线性的**，所以还是普通线性回归，只是换了特征。这就是多项式回归。
A linear model $\hat y = w_0 + w_1 x$ only draws a line. But if we **expand $x$ into $[x, x^2, x^3]$ first, then do linear regression**, the model becomes $\hat y = w_0 + w_1 x + w_2 x^2 + w_3 x^3$ — a curve! **Key insight: the model is still linear in the parameters $\mathbf{w}$**, so it's still ordinary linear regression, just with new features. That's polynomial regression.

**阶数(degree)** 控制曲线的弯曲程度，也就是**模型复杂度**——这正是演示过拟合的完美旋钮。我们造一份真实关系是 3 次曲线的数据。
The **degree** controls how wiggly the curve is — i.e. **model complexity** — the perfect knob for demonstrating overfitting. We make data whose true relationship is a cubic.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score, validation_curve, learning_curve
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(42)

# 真实关系是 3 次曲线, 加噪声 / true relationship is cubic + noise
def true_f(x): return 0.5*x**3 - 2*x**2 + x + 3
n = 80
x = np.sort(rng.uniform(-2, 4, n))
y = true_f(x) + rng.normal(0, 3, n)        # 真值 + 噪声 σ=3
X = x.reshape(-1, 1)
print(f"真实函数 true: 0.5x³-2x²+x+3 (3阶), 噪声 σ=3, n={n}")


<a id="2"></a>
## 2. 欠拟合/恰好/过拟合 ⭐ / Under / Just-right / Over

用三个阶数直观对比（用 Pipeline 把多项式特征+缩放+回归打包，防泄漏 3.12）：
Three degrees side by side (Pipeline bundles poly-features + scaling + regression, leak-free per 3.12):
- **阶=1（欠拟合）**：直线拟合曲线，**train 和 CV 都差**——模型太简单，**高偏差**。
  **degree=1 (underfit):** a line fits a curve; **train and CV both poor** — too simple, **high bias**.
- **阶=3（恰好）**：匹配真实复杂度，**train 和 CV 都好且接近**。
  **degree=3 (just right):** matches the true complexity; **train and CV both good and close**.
- **阶=15（过拟合）**：曲线扭来扭去去贴每个噪声点，**train 极好但 CV 崩**——**高方差**。
  **degree=15 (overfit):** the curve wiggles to chase every noisy point; **train excellent but CV collapses** — **high variance**.

**记住这个签名：train 远好于 CV = 过拟合；两者都差 = 欠拟合。**
**Remember the signature: train ≫ CV = overfitting; both poor = underfitting.**


In [ ]:
x_plot = np.linspace(-2, 4, 300).reshape(-1, 1)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, deg, title in [(axes[0], 1, "欠拟合 underfit (阶=1)"),
                       (axes[1], 3, "恰好 just-right (阶=3)"),
                       (axes[2], 15, "过拟合 overfit (阶=15)")]:
    # 多项式特征 → 标准化 → 线性回归, 打包成 Pipeline / poly features then linear regression
    model = make_pipeline(PolynomialFeatures(deg), StandardScaler(), LinearRegression()).fit(X, y)
    train_r2 = model.score(X, y)                                  # 训练集 R²
    cv_r2 = cross_val_score(model, X, y, cv=5, scoring="r2").mean()  # 交叉验证 R²
    ax.scatter(x, y, alpha=0.4, s=15)
    ax.plot(x_plot, true_f(x_plot), "g--", lw=1.5, label="真实 true")
    ax.plot(x_plot, model.predict(x_plot), "r-", lw=2, label=f"拟合 fit (阶={deg})")
    ax.set_ylim(y.min()-5, y.max()+5); ax.legend(fontsize=8)
    ax.set_title(f"{title}\ntrain R²={train_r2:.2f}, CV R²={cv_r2:.2f}")
plt.tight_layout(); plt.show()
print("欠拟合: train/CV 都差(高偏差); 恰好: 都好且接近; 过拟合: train>>CV(追噪声, 高方差)")


<a id="3"></a>
## 3. 偏差-方差权衡（实验拆解）⭐ / Bias-Variance Trade-off

任何模型的预测误差都能分解成三部分：
Any model's prediction error decomposes into three parts:

$$\text{误差} = \underbrace{\text{偏差}^2}_{\text{系统性偏离真值}} + \underbrace{\text{方差}}_{\text{对训练数据的敏感度}} + \underbrace{\text{不可约噪声}}_{\text{数据本身的随机性}}$$

- **偏差(bias)**：模型太简单，系统性地偏离真实规律（欠拟合）。
  **Bias:** the model is too simple and systematically misses the truth (underfitting).
- **方差(variance)**：模型太复杂，对训练数据的随机波动过度敏感（换一批数据就给出很不同的预测，过拟合）。
  **Variance:** the model is too complex and over-sensitive to training noise (a different sample gives very different predictions, overfitting).

**关键权衡**：复杂度↑ → 偏差↓ 但方差↑。总误差是 **U 形**，最优在中间。下面用"重复采样多次、在固定测试点上测预测的均值和波动"来**实测**这条曲线。
**The trade-off:** complexity↑ → bias↓ but variance↑. Total error is **U-shaped**, optimal in the middle. Below we **measure** this curve by resampling many times and tracking the mean and spread of predictions at fixed test points.


In [ ]:
x_test = np.linspace(-1.5, 3.5, 50).reshape(-1, 1)
y_test_true = true_f(x_test).ravel()      # 测试点的真值(无噪声)
degrees_range = range(1, 13)
n_repeats = 100                            # 每个阶数重采样 100 次

bias2_list, var_list, total_list = [], [], []
for deg in degrees_range:
    preds = np.zeros((n_repeats, len(x_test)))
    for r in range(n_repeats):
        xr = rng.uniform(-2, 4, n); yr = true_f(xr) + rng.normal(0, 3, n)   # 重采样训练集
        m = make_pipeline(PolynomialFeatures(deg), StandardScaler(), LinearRegression()).fit(xr.reshape(-1,1), yr)
        preds[r] = m.predict(x_test)
    bias2 = np.mean((preds.mean(0) - y_test_true)**2)   # 偏差²: 平均预测偏离真值多少
    var = np.mean(preds.var(0))                          # 方差: 不同采样间预测的波动
    bias2_list.append(bias2); var_list.append(var); total_list.append(bias2+var)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(list(degrees_range), bias2_list, "o-", label="偏差² Bias² (模型太简单)")
ax.plot(list(degrees_range), var_list, "s-", label="方差 Variance (模型太敏感)")
ax.plot(list(degrees_range), total_list, "^-", lw=2, label="总误差 Bias²+Var")
ax.axvline(3, color="g", ls="--", alpha=0.5, label="真实阶数 true=3")
ax.set_xlabel("多项式阶数(复杂度) degree"); ax.set_ylabel("error"); ax.set_yscale("log"); ax.legend()
ax.set_title("偏差-方差权衡: 总误差 U 形, 最优在中等复杂度")
plt.tight_layout(); plt.show()
print("阶数低→高偏差(欠拟合); 阶数高→高方差(过拟合); 总误差 U 形, 最优≈真实阶数 3")


<a id="4"></a>
## 4. 验证曲线：选复杂度 ⭐ / Validation Curve

**验证曲线**：横轴是某个超参（这里是阶数），纵轴是 train 和 CV 的分数。它把"过拟合的边界"画了出来：**train 分数随复杂度单调上升**（总能更贴训练点），但 **CV 分数先升后降**——CV 的峰值就是最优复杂度。这是调单个超参的标准工具。
**Validation curve:** x-axis is a hyperparameter (here the degree), y-axis is train and CV scores. It draws the overfitting boundary: **train rises monotonically with complexity** (always fits training data better), but **CV rises then falls** — CV's peak is the optimal complexity. The standard tool for tuning one hyperparameter.


In [ ]:
degrees_v = np.arange(1, 16)
# validation_curve 自动对每个 degree 做 CV, 返回 train 和验证分数 / sweep one hyperparameter
train_scores, val_scores = validation_curve(
    make_pipeline(PolynomialFeatures(), StandardScaler(), LinearRegression()),
    X, y, param_name="polynomialfeatures__degree", param_range=degrees_v, cv=5, scoring="r2")

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(degrees_v, train_scores.mean(1), "o-", label="train R²")
ax.plot(degrees_v, val_scores.mean(1), "s-", label="CV R²")
ax.fill_between(degrees_v, val_scores.mean(1)-val_scores.std(1), val_scores.mean(1)+val_scores.std(1), alpha=0.2)
best_deg = degrees_v[np.argmax(val_scores.mean(1))]
ax.axvline(best_deg, color="r", ls="--", label=f"最优阶数 best={best_deg}")
ax.set_xlabel("degree"); ax.set_ylabel("R²"); ax.set_ylim(0, 1.05); ax.legend()
ax.set_title("验证曲线: train↑ 但 CV 先升后降, 选 CV 峰值")
plt.tight_layout(); plt.show()
print(f"CV 选出最优阶数 = {best_deg} (接近真实 3)")
print("train R² 随阶数单调升(总能更贴训练点), CV 先升后降 → 这就是过拟合的边界")


<a id="5"></a>
## 5. 学习曲线：诊断 + 对症下药 ⭐ / Learning Curve

**学习曲线**：横轴是**训练样本数**，看 train 和 CV 分数随数据增多怎么变。它能区分高偏差和高方差，并告诉你**该怎么救**（极其实用）：
**Learning curve:** x-axis is the **number of training samples**, showing how train and CV scores evolve as data grows. It tells high-bias from high-variance and **how to fix each** (extremely practical):
- **高偏差**：train 和 CV **都低且早早收敛到一起**。→ **加数据没用**，要更复杂的模型/更多特征。
  **High bias:** train and CV are **both low and converge early**. → **More data won't help**; need a more complex model / more features.
- **高方差**：train 高、CV 低、**两者之间有大缺口**。→ **加数据能缩小缺口**（或降复杂度/加正则 4.4）。
  **High variance:** train high, CV low, **a big gap between them**. → **More data closes the gap** (or reduce complexity / add regularization, 4.4).


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, deg, title in [(axes[0], 1, "高偏差 high-bias (阶=1, 欠拟合)"),
                       (axes[1], 15, "高方差 high-variance (阶=15, 过拟合)")]:
    # learning_curve 在不同训练样本量下做 CV / scores vs training-set size
    sizes, tr, va = learning_curve(
        make_pipeline(PolynomialFeatures(deg), StandardScaler(), LinearRegression()),
        X, y, cv=5, scoring="r2", train_sizes=np.linspace(0.2, 1.0, 8))
    ax.plot(sizes, tr.mean(1), "o-", label="train")
    ax.plot(sizes, va.mean(1), "s-", label="CV")
    ax.set_xlabel("训练样本数 #train samples"); ax.set_ylabel("R²"); ax.legend(); ax.set_title(title)
plt.tight_layout(); plt.show()
print("左(高偏差): train/CV 都低且收敛 → 加数据没用, 要更复杂模型/更多特征")
print("右(高方差): train 高 CV 低有大缺口 → 加数据能缩小缺口(或减复杂度/加正则 4.4)")


<a id="6"></a>
## 6. 小结 / Summary

```
多项式回归 = 把 x,x²,x³... 当特征做线性回归(对参数仍线性); 阶数=复杂度旋钮
欠拟合(高偏差): train/CV 都差; 过拟合(高方差): train>>CV; 恰好: 都好且接近
偏差-方差: 误差=偏差²+方差+噪声; 复杂度↑→偏差↓方差↑; 总误差 U 形, 最优在中间
验证曲线(横轴=超参): train 单调升, CV 先升后降 → 选 CV 峰值
学习曲线(横轴=样本数): 高偏差(都低早收敛, 加数据没用) vs 高方差(大缺口, 加数据/正则有用)
```

### 💡 面试速查 / Interview cheat-sheet
1. **过拟合 = train≫CV(高方差)**; **欠拟合 = 都差(高偏差)**。
   Overfit = train≫CV (high variance); underfit = both poor (high bias).
2. **偏差-方差权衡**: 复杂度↑→偏差↓方差↑, 总误差 U 形。
   Bias-variance: complexity↑ → bias↓ variance↑, total error U-shaped.
3. **验证曲线选复杂度**(CV 峰值); **学习曲线诊断偏差/方差**。
   Validation curve picks complexity; learning curve diagnoses bias/variance.
4. **高偏差加数据没用**(要更复杂模型); **高方差加数据/正则有用**。
   High bias: more data won't help (need complexity); high variance: more data/regularization helps.
5. 多项式回归**对参数线性**, 所以仍是线性回归。
   Polynomial regression is linear in parameters, so still linear regression.

### 下一节 / Next
**4.4 岭回归(Ridge)**——过拟合的解药之一: 给系数加 L2 惩罚, 把它们"拉小", 用一点偏差换大量方差下降, 还能治多重共线性。
**4.4 Ridge** — a cure for overfitting: add an L2 penalty to shrink coefficients, trading a little bias for a lot of variance reduction, and curing multicollinearity.
